In [5]:
import numpy as np
import pandas as pd
import geopandas as gpd
import json
import time
import requests


BASE_URL='https://api.nyc.gov/geoclient/v2/'
API_KEY = "26c2d3b118074d08aec7c054b87d3eae"


with open('../data/google_api.txt', 'r') as file:
    GOOGLE_API_KEY = file.read()


cached = pd.read_csv('../data/cached.csv')

In [6]:
rl = pd.read_csv("https://data.cityofnewyork.us/resource/pvqr-7yc4.csv?$where=violation_code=7&$limit=999999999999999999")

rl.to_csv("../data/rl.csv")

<positron-console-cell-6>:1: DtypeWarning: Columns (0: violation_in_front_of_or_opposite, 1: days_parking_in_effect, 2: from_hours_in_effect, 3: to_hours_in_effect, 4: meter_number) have mixed types. Specify dtype option on import or set low_memory=False.


In [7]:
mapping = {
    "BK": "Brooklyn",
    "BX": "Bronx",
    "MN": "Manhattan",
    "QN": "Queens",
    "ST": "Staten Island"
}

rl["boro"] = rl["violation_county"].map(mapping).fillna("New York")

rl['intersecting_street'] = rl['intersecting_street'].fillna("")

rl["full_street"] = rl['street_name'] + rl['intersecting_street']


In [8]:

# Replace bad substrings
replacements_sub = {
    "(S/B)": "",
    "(N/B)": "",
    "(W/B)": "",
    "(E/B)": "",
    "SB ": "",
    "WB ": "",
    "EB ": "",
    "NB ": "",
    "THST": "TH ST",
    "STST": "ST ST",
    "NDST": "ND ST",
    "RDST": "RD ST",
    "EXPWYEXT": "EXPWY",
    "DOLEST": "DOLE ST",
    "THAVE": "TH AVE",
    "STAVE": "ST AVE",
    "NDAVE": "ND AVE",
    "RDAVE": "RD AVE",
    "AVEK": "AVE K",
    "AVEO": "AVE O",
    "BAYPKWY": "BAY PKWY",
    "PKWYW": "PKWY W",
    "MAINST": "MAIN ST",
    "BOUCKCT": "BOUCK CT",
    "BRONXSTATE": "BRONX STATE",
    "WARRENST": "WARREN ST",
    "UNION TRPK": "UNION TPKE",
    "KENTST": "KENT ST",
    "BENSONAVE": "BENSON AVE",
    "RIVERPKWY": "RIVER PKWY",
    "PKWYE": "PKWY E",
    "SAGEST": "SAGE ST",
    "REEVESAVE": "REEVES AVE",
    "SPORTLAND": "S PORTLAND",
    "VICTORYBLVD": "VICTORY BLVD",
    "JEWELAVE": "JEWEL AVE",
    "ADAMCLAYTON": "ADAM CLAYTON",
    "WBROADWAY": "W BROADWAY",
    "GR CENTRAL": "GRAND CENTRAL",
    "GRANDCENTRAL": "GRAND CENTRAL",
    "MCDONALDAVE": "MCDONALD AVE",
    "BATTERYAVE": "BATTERY AVE",
    "SUTTERAVE": "SUTTER AVE",
    "DUNKIRK AVE": "DUNKIRK ST",
    "RHINEAVE": "RHINE AVE",
    "CAPODANNOBLVD": "CAPODANNO BLVD",
    "ELRIDGE": "ELDRIDGE",
    "BAYTERRACE": "BAY TERRACE",
    "ROSEAVE": "ROSE AVE",
    "CARVER PL": "CARVER LOOP",
    "WKINGSBRIDGE": "W KINGSBRIDGE",
    "RDRD": "RD RD",
    "YOUNGAVE": "YOUNG AVE",
    "AVEL": "AVE L",
    "KINGST": "KING ST",
    "MARYSAVE": "MARYS AVE",
    "JUNIPERVALLEY": "JUNIPER VALLEY",
    "CHERRYST": "CHERRY ST",
    "EBUCHANAN": "E BUCHANAN",
    "NOMEAVE": "NOME AVE",
    "FERNDALE ST": "FERNDALE AVE",
    "LUDWIGST": "LUDWIG ST",
    "ALLENAVE": "ALLEN AVE",
    "OLDBROADWAY": "OLD BROADWAY",
    "DOWNINGST": "DOWNING ST",
    "BRONX RIVER PKWY R": "BRONX RIVER PKWY",
    "KING STON": "KINGSTON",
    "ASTORIA BLVD S @ 77TH ST": "ASTORIA BLVD @ 77TH ST",
    "BRONX RIVER PKWY OFFRAMP": "BRONX RIVER PKWY",
    "GRAND CENTRAL PKWY SVC RD": "GRAND CENTRAL PKWY",
    "E/B BR": "BRUCKNER BLVD",
    "WILLIAMSBURG ST": "WILLIAMSBURG ST W",
    "WHITESTONE EXPWY SVCRD": "WHITESTONE EXPWY",
    "L.I.E. N. SVC ROAD": "LONG ISLAND EXPY",
    "70TH ST / 45TH AVE": "70TH ST",
    "ENEW YORK AVE": "E NEW YORK AVE",
    "LENOXRD": "LENOX RD",
    "CLERMONT PKWY": "CLAREMONT PKWY",
    "STATEN ISLAND EXPRESSWAY EXIT 12": "REON AVE",
    "HORACEHARDING": "HORACE HARDING",
    "VAN CORTLAND AVE W": "VAN CORTLANDT AVE W",
    "REIDAVE": "REID AVE",
    "NEW YORKAVE": "NEW YORK AVE",
    "WILLIAMSBURG ST W W": "WILLIAMSBURG ST W",
    "GRAND AVE / BROADWAY": "GRAND AVE",
    "UTICAAVE": "UTICA AVE",
    "VAN WYCK EXPWY / E": "VAN WYCK EXPWY",
    "HOOK CREEK BLVD @ST JOHNS AVE":  "OCEAN AVE @ 130TH AVE",
    "71ST AVE / CONTINENTAL": "71ST AVE",
    "NREMSEN AVE": "REMSEN AVE",
    "GUY RBREWER BLVD": "GUY R BREWER BLVD",
    "E NEWYORK AVE": "E NEW YORK AVE",
    "UNIONTPKE": "UNION TPKE",
    "FOSTER AVE @ E NEW YORK AVE": "FOSTER AVE @ NEW YORK AVE",
    "NEWDORP LN": "NEW DORP LN",
    "GLENST": "GLEN ST",
}

for old, new in replacements_sub.items():
    rl['full_street'] = rl['full_street'].str.replace(old, new)

rl[['street1', 'street2']] = (
    rl['full_street']
    .str.split(r'[@]', n=1, expand=True)
    .apply(lambda col: col.str.strip())
)

replacements_streets = {
    "AVEN": "AVE N",
    "AVES": "AVE S",
    "MANHATTAN COLLE": "MANHATTAN COLLEGE PKWY",
    "HOYT AVE S": "HOYT AVE N",
    "ADAM CLAYTON POWELL JR B": "ADAM CLAYTON POWELL JR BLVD",
    "TILLARY S": "TILLARY ST",
    "AVIATOR SPORTS COMP": "AVIATION RD",
    "SL": "SLOSSON AVE",
    "AVEU": "AVE U",
    "AVEM": "AVE M",
    "N CONDUIT": "N CONDUIT AVE",
    "PARK AND R": "PARK AND RIDE",
    "CADMAN PLAZA": "CADMAN PLAZA W",
    "DRUMGOOLE RD": "DRUMGOOLE RD W",
    "GUYR BREWER BLVD": "GUY R BREWER BLVD",
    "AVEY": "AVE Y",
    "BROOKLYNAVE": "BROOKLYN AVE",
    "HERBERTON AVE": "HEBERTON AVE",
    "DRIGGSAVE": "DRIGGS AVE",
    "EFORDHAM RD": "E FORDHAM RD",
    "EGUN HILL RD": "E GUN HILL RD",
    "ROCKAWAYAVE": "ROCKAWAY AVE",
    "GERRITSENAVE": "GERRITSEN AVE"
}

rl['street1'] = rl['street1'].replace(replacements_streets)
rl['street2'] = rl['street2'].replace(replacements_streets)


rl_adds = (
    rl.groupby(['street1', 'street2', 'boro'], as_index=False, dropna=False)
      .size()
)

rl_adds = rl_adds[rl_adds['size'] > 25].reset_index(drop=True)

In [9]:
joined = rl_adds.merge(
    cached,
    on=["street1", "street2", "boro"],
    how="left"
)

# Rows that successfully matched
matched = joined[joined["lat"].notna() & joined["lon"].notna()].copy()

# Rows that did not match
unmatched = joined[joined["lat"].isna() | joined["lon"].isna()].copy()

# Keep only desired columns
matched = matched[
    ["street1", "street2", "boro", "size", "lat", "lon"]
]

unmatched = unmatched[
    ["street1", "street2", "boro", "size"]
]

unmatched["id"] = unmatched.index

In [ ]:

i = 1681

data_url = f'{BASE_URL}intersection.json?crossStreetOne={rl_adds['street1'][i]}&crossStreetTwo={rl_adds['street2'][i]}&borough={rl_adds['boro'][i]}&subscription-key={API_KEY}'
data_url = f'{BASE_URL}intersection.json?crossStreetOne=QUEENS MIDTOWN EXPWY&crossStreetTwo=69TH LN&borough={rl_adds['boro'][i]}&compassDirection=N&subscription-key={API_KEY}'
response = requests.get(data_url)

package=response.json()

# package['intersection']['latitude']


In [10]:
coded=[]
bunk=[]


# loop through each address to search
for i in range(0, len(unmatched)):
    
    # pull using NYC GeoSearch
    data_url = f'{BASE_URL}intersection.json?crossStreetOne={unmatched['street1'][i]}&crossStreetTwo={unmatched['street2'][i]}&borough={unmatched['boro'][i]}&compassDirection=N&subscription-key={API_KEY}'
    response = requests.get(data_url)
    data = response.json()
    
    if 'latitude' in data['intersection']:
        coded.append([data['intersection']['latitude'], data['intersection']['longitude'], unmatched['id'][i]])
        continue
    
    if data['intersection']['message'] == "COMPASS DIRECTION VALUE IS INVALID FOR THIS INPUT LOCATION":
        data_url = f'{BASE_URL}intersection.json?crossStreetOne={unmatched['street1'][i]}&crossStreetTwo={unmatched['street2'][i]}&borough={unmatched['boro'][i]}&compassDirection=E&subscription-key={API_KEY}'
        response = requests.get(data_url)
        data = response.json()
        if 'latitude' in data['intersection']:
            coded.append([data['intersection']['latitude'], data['intersection']['longitude'], unmatched['id'][i]])
            continue

    bunk.append([unmatched['street1'][i], unmatched['street2'][i], unmatched['boro'][i], unmatched['id'][i], data['intersection']['message'], unmatched['size'][i]])


bunk_df = pd.DataFrame(bunk, columns = ["street1", "street2", "boro", "id", "message", "size"])
coded_df = pd.DataFrame(coded, columns = ["lat", "lon", "id"])

In [11]:

def geocode_intersection(row):
    # Build intersection query
    address = f"{row['street1']} & {row['street2']}, {row['boro']}, New York, NY"
    
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    
    params = {
        "address": address,
        "key": GOOGLE_API_KEY
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        # Check API status
        if data["status"] != "OK" or len(data["results"]) == 0:
            return {
                "id": row["id"],
                "status": data.get("status"),
                "lat": None,
                "lon": None
            }
        
        # Take best result
        result = data["results"][0]
        location = result["geometry"]["location"]
        
        return {
            "id": row["id"],
            "status": "OK",
            "lat": location["lat"],
            "lon": location["lng"]
        }
    
    except Exception as e:
        return {
            "id": row["id"],
            "status": str(e),
            "lat": None,
            "lon": None
        }


# ---- APPLY TO DATAFRAME ---- #

results = []

for _, row in bunk_df.iterrows():
    res = geocode_intersection(row)
    results.append(res)
    
    # IMPORTANT: rate limiting (Google free tier ~50 QPS but be safe)
    time.sleep(0.05)

results_df = pd.DataFrame(results)

# ---- SPLIT INTO GOOD / BAD ---- #
if results_df.empty:
  good_df = pd.DataFrame()
  bad_df = pd.DataFrame()
else:
  good_df = results_df[results_df["status"] == "OK"][["id", "lat", "lon"]]
  bad_df = results_df[results_df["status"] != "OK"]

In [ ]:
# create a table of results (address, lat, lon)



In [12]:
coords_df = pd.concat([good_df, coded_df])

rl_adds_latlon = pd.concat([matched, unmatched.merge(coords_df, on = "id")])

In [13]:


rl_full = rl.merge(rl_adds_latlon, on = ["street1", "street2", "boro"])

rl_full.to_csv("../data/rl_merged.csv")

rl_adds_latlon.drop(columns = ['size', 'id']).to_csv("../data/cached.csv")